In [6]:
from data_loader import Phase
import torch

In [9]:
from torch.utils.data import DataLoader, RandomSampler, TensorDataset

In [44]:
import torch

# Assuming you have the dictionary containing probabilities
# For example:
identifier_probabilities = {0: 0.2, 1: 0.4, 2: 0.2, 4: 0.2}

# Step 1: Convert the probabilities to a tensor
probabilities_tensor = torch.tensor(list(identifier_probabilities.values()))

# Step 2: Sample an index using multinomial based on the probabilities
selected_loader_idx = torch.multinomial(probabilities_tensor, 1).item()

# Step 3: Retrieve the identifier associated with the sampled index
selected_identifier = list(identifier_probabilities.keys())[selected_loader_idx]

print(selected_identifier)



1


In [29]:
dummy_tensor = torch.tensor([0,1,2,4,1,])
unique_identifiers, identifier_counts = torch.unique(dummy_tensor, return_counts=True)
identifier_count_dict = {identifier.item(): count.item() / len(dummy_tensor) for identifier, count in zip(unique_identifiers, identifier_counts)}


In [26]:
identifier_count_dict = {identifier.item(): count.item() for identifier, count in zip(unique_identifiers, identifier_counts)}


In [30]:
identifier_count_dict

{0: 0.2, 1: 0.4, 2: 0.2, 4: 0.2}

In [80]:
class MetaDataLoader:
    def __init__(self, data_loaders):
        self.data_loaders = data_loaders
        self.num_loaders = len(data_loaders)
        self.loaders = [iter(loader) for loader in data_loaders.values()]
        self.num_batches_fetched = 0

    def __len__(self):
        return sum(len(loader) for loader in self.data_loaders.values())

    def __iter__(self):
        return self

    def __next__(self):
        if self.num_batches_fetched == len(self):
            # All batches have been fetched, raise StopIteration
            raise StopIteration

        selected_loader_idx = self.num_batches_fetched % self.num_loaders
        try:
            batch = next(self.loaders[selected_loader_idx])
        except StopIteration:
            # Reset the iterator for the selected loader if it's exhausted
            self.loaders[selected_loader_idx] = iter(self.data_loaders[selected_loader_idx])
            batch = next(self.loaders[selected_loader_idx])

        self.num_batches_fetched += 1
        return batch


import itertools

class RandomMetaDataLoader:
    def __init__(self, data_loaders,identifier_prob_dict):
        self.data_loaders = data_loaders
        self.num_loaders = len(data_loaders)
        self.loaders = [iter(loader) for loader in data_loaders.values()]
        self.num_batches_fetched = 0
        self.identifier_prob_dict = identifier_prob_dict
        

    def __len__(self):
        return sum(len(loader) for loader in self.data_loaders.values())


    def __iter__(self):
        return self

    def __next__(self):
        if self.num_batches_fetched == len(self):
            # All batches have been fetched, raise StopIteration
            raise StopIteration

        # Sample an index using multinomial based on the probabilities
        selected_loader_idx = torch.multinomial(torch.tensor(list(self.identifier_prob_dict.values())), 1).item()

        # Retrieve the identifier associated with the sampled index
        selected_identifier = list(self.identifier_prob_dict.keys())[selected_loader_idx]

        try:
            batch = next(self.loaders[selected_identifier])
        except StopIteration:
            # Reset the iterator for the selected loader if it's exhausted
            self.loaders[selected_identifier] = iter(self.data_loaders[selected_identifier])
            batch = next(self.loaders[selected_identifier])

        self.num_batches_fetched += 1
        return batch
    

def create_data_loader(data_path, batch_size, phase, sequential=False,random_sequence=True):
    phase_str = phase.value

    data_inputs, data_labels = torch.load(inputs_path), torch.load(label_path)
    identifier_tensor = torch.load(identifier_path)  # Load the identifier tensor

    data_inputs = torch.rand((num_samples, num_features))
    data_labels = torch.randint(2, (num_samples,))
    identifier_tensor = torch.randint(num_tasks, (num_samples,))

    dataset = TensorDataset(data_inputs, data_labels, identifier_tensor)  # Include the identifier tensor in the dataset
    
    if sequential:

        # If sequential=True, create a dictionary of dataloaders for each unique identifier value
        unique_identifiers, identifier_counts = torch.unique(identifier_tensor, return_counts=True)
        identifier_prob_dict = {identifier.item(): count.item() / len(identifier_tensor) for identifier, count in zip(unique_identifiers, identifier_counts)}
        data_loaders = {}

        for identifier in unique_identifiers:
            indices = torch.nonzero(identifier_tensor == identifier).squeeze()
            subset_dataset = torch.utils.data.Subset(dataset, indices)
            data_loaders[identifier.item()] = DataLoader(subset_dataset, batch_size=batch_size, shuffle=False)

        if random_sequence:
            return RandomMetaDataLoader(data_loaders, identifier_prob_dict)
        else:
            return MetaDataLoader(data_loaders)

    else:
        if phase_str == Phase.TRAIN:
            sampler = RandomSampler(dataset)
        else:
            sampler = None
        return DataLoader(dataset, sampler=sampler, batch_size=batch_size)


In [81]:
sequential_loader = create_data_loader(data_path="", batch_size=64, phase=Phase.TRAIN, sequential=True,random_sequence=True)
simultaneous_loader = create_data_loader(data_path="", batch_size=64, phase=Phase.TRAIN, sequential=False,random_sequence=True)

In [82]:
from collections import Counter

In [83]:
for i, data in enumerate(sequential_loader):
    print(f"Batch number {i}")
    value_count = Counter(data[2].tolist())
    for value, count in value_count.items():
        print(f"Value {value}: Count {count}")
    print("--------------")

Batch number 0
Value 0: Count 64
--------------
Batch number 1
Value 1: Count 64
--------------
Batch number 2
Value 2: Count 64
--------------
Batch number 3
Value 3: Count 64
--------------
Batch number 4
Value 4: Count 64
--------------
Batch number 5
Value 0: Count 64
--------------
Batch number 6
Value 1: Count 64
--------------
Batch number 7
Value 2: Count 64
--------------
Batch number 8
Value 3: Count 64
--------------
Batch number 9
Value 4: Count 64
--------------
Batch number 10
Value 0: Count 64
--------------
Batch number 11
Value 1: Count 64
--------------
Batch number 12
Value 2: Count 42
--------------
Batch number 13
Value 3: Count 64
--------------
Batch number 14
Value 4: Count 64
--------------
Batch number 15
Value 0: Count 12
--------------
Batch number 16
Value 1: Count 14
--------------
Batch number 17
Value 2: Count 64
--------------
Batch number 18
Value 3: Count 34
--------------


In [79]:
for i, data in enumerate(simultaneous_loader):
    print(f"Batch number {i}")
    value_count = Counter(data[2].tolist())
    for value, count in value_count.items():
        print(f"Value {value}: Count {count}")
    print("--------------")

Batch number 0
Value 3: Count 16
Value 1: Count 13
Value 0: Count 16
Value 4: Count 9
Value 2: Count 10
--------------
Batch number 1
Value 3: Count 16
Value 2: Count 10
Value 1: Count 13
Value 4: Count 15
Value 0: Count 10
--------------
Batch number 2
Value 4: Count 14
Value 2: Count 14
Value 3: Count 14
Value 0: Count 10
Value 1: Count 12
--------------
Batch number 3
Value 2: Count 18
Value 1: Count 14
Value 0: Count 10
Value 3: Count 14
Value 4: Count 8
--------------
Batch number 4
Value 1: Count 20
Value 3: Count 11
Value 0: Count 15
Value 4: Count 10
Value 2: Count 8
--------------
Batch number 5
Value 4: Count 18
Value 3: Count 11
Value 0: Count 12
Value 2: Count 16
Value 1: Count 7
--------------
Batch number 6
Value 4: Count 11
Value 3: Count 16
Value 0: Count 16
Value 1: Count 8
Value 2: Count 13
--------------
Batch number 7
Value 3: Count 21
Value 1: Count 11
Value 2: Count 9
Value 0: Count 13
Value 4: Count 10
--------------
Batch number 8
Value 2: Count 14
Value 0: Coun